In [1]:
# ==============================================================================
# CELL 1: REGISTER PAYSIM CSV AS BRONZE DELTA TABLE
# ==============================================================================
from pyspark.sql import functions as F

# Reading exact filename uploaded from F:\FinIntel360\RawData\paysim.csv.csv
raw_path = "Files/raw_paysim/paysim.csv.csv"

df_paysim_raw = spark.read.option("header", "true").option("inferSchema", "true").csv(raw_path)

df_paysim_bronze = df_paysim_raw \
    .withColumn("tx_id", F.monotonically_increasing_id()) \
    .withColumn("ingested_at", F.current_timestamp())

df_paysim_bronze.write.format("delta").mode("overwrite").saveAsTable("bronze_paysim_tx")

total_rows = df_paysim_bronze.count()
fraud_rows = df_paysim_bronze.filter(F.col("isFraud") == 1).count()
print(f"Registered bronze_paysim_tx: {total_rows:,} total transactions | {fraud_rows:,} confirmed fraud cases ({fraud_rows/total_rows*100:.3f}%)")

StatementMeta(, 1e5b9eeb-8936-4921-af84-2d7f59763fe5, 3, Finished, Available, Finished, False)

Registered bronze_paysim_tx: 6,362,620 total transactions | 8,213 confirmed fraud cases (0.129%)


In [2]:
# ==============================================================================
# CELL 2: SILVER LAYER - FRAUD FEATURE ENGINEERING (6.36M ROWS)
# ==============================================================================
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_bronze = spark.read.table("bronze_paysim_tx")

# 1. Accounting Violation & Balance Drain Features
df_features = df_bronze \
    .withColumn("errorBalanceOrig", F.round(F.col("newbalanceOrig") - (F.col("oldbalanceOrg") - F.col("amount")), 2)) \
    .withColumn("errorBalanceDest", F.round(F.col("newbalanceDest") - (F.col("oldbalanceDest") + F.col("amount")), 2)) \
    .withColumn("origBalanceDrainRatio", F.when(F.col("oldbalanceOrg") > 0, F.round(F.col("amount") / F.col("oldbalanceOrg"), 4)).otherwise(1.0)) \
    .withColumn("isZeroBalanceOrigAfter", F.when((F.col("oldbalanceOrg") > 0) & (F.col("newbalanceOrig") == 0), 1).otherwise(0)) \
    .withColumn("isZeroBalanceDestAfter", F.when((F.col("amount") > 0) & (F.col("newbalanceDest") == 0), 1).otherwise(0)) \
    .withColumn("isHighRiskChannel", F.when(F.col("type").isin("TRANSFER", "CASH_OUT"), 1).otherwise(0)) \
    .withColumn("hourOfDay", F.col("step") % 24)

# 2. Rolling 1-Hour Window Velocity Metrics per Origin & Destination Account
w_orig_1h = Window.partitionBy("nameOrig").orderBy("step").rangeBetween(-1, 0)
w_dest_1h = Window.partitionBy("nameDest").orderBy("step").rangeBetween(-1, 0)

df_silver_tx = df_features \
    .withColumn("orig_tx_count_1h", F.count("tx_id").over(w_orig_1h)) \
    .withColumn("orig_tx_sum_1h", F.round(F.sum("amount").over(w_orig_1h), 2)) \
    .withColumn("dest_inflow_count_1h", F.count("tx_id").over(w_dest_1h)) \
    .withColumn("dest_inflow_sum_1h", F.round(F.sum("amount").over(w_dest_1h), 2))

# Partition by transaction type for fast Direct Lake & ML filtering
df_silver_tx.write.format("delta").mode("overwrite").partitionBy("type").saveAsTable("silver_paysim_enriched")
print(f"Created silver_paysim_enriched with {df_silver_tx.count():,} rows and {len(df_silver_tx.columns)} engineered features.")

StatementMeta(, 1e5b9eeb-8936-4921-af84-2d7f59763fe5, 4, Finished, Available, Finished, False)

Created silver_paysim_enriched with 6,362,620 rows and 24 engineered features.


In [4]:
# ==============================================================================
# CELL 3 (FIXED): SILVER LAYER - ECONOMETRIC MACRO & YIELD CURVE FEATURES
# ==============================================================================
import pandas as pd
import numpy as np
from pyspark.sql import functions as F

pdf_fred = spark.read.table("bronze_fred_macro").toPandas()

# Explicitly cast observation_date to Pandas datetime64 so resample() gets a DatetimeIndex
pdf_fred["observation_date"] = pd.to_datetime(pdf_fred["observation_date"])

# Pivot series into wide format by observation_date
pdf_wide = pdf_fred.pivot_table(index="observation_date", columns="series_id", values="value").sort_index()

# Forward-fill daily/monthly frequencies so all indicators align cleanly
pdf_wide = pdf_wide.ffill().dropna()

# Resample to Quarterly (Quarter-End) for corporate revenue regression alignment
pdf_q = pdf_wide.resample("QE").last().reset_index()

# Compute Econometric Covariates
pdf_q["yield_spread_10y2y"] = (pdf_q["DGS10"] - pdf_q["DGS2"]).round(4)
pdf_q["yield_curve_inverted"] = (pdf_q["yield_spread_10y2y"] < 0).astype(int)
pdf_q["cpi_yoy_pct"] = (pdf_q["CPIAUCSL"].pct_change(4) * 100).round(4)
pdf_q["fed_funds_diff1"] = pdf_q["FEDFUNDS"].diff(1).round(4)

# Approximate Nelson-Siegel Level (Long rate) & Slope (Short - Long)
pdf_q["ns_level"] = pdf_q["DGS10"].round(4)
pdf_q["ns_slope"] = (pdf_q["FEDFUNDS"] - pdf_q["DGS10"]).round(4)

pdf_q = pdf_q.dropna().reset_index(drop=True)
pdf_q["fiscal_year"] = pdf_q["observation_date"].dt.year.astype(int)
pdf_q["fiscal_quarter"] = "Q" + pdf_q["observation_date"].dt.quarter.astype(str)

sdf_silver_macro = spark.createDataFrame(pdf_q) \
    .withColumn("observation_date", F.to_date("observation_date"))

sdf_silver_macro.write.format("delta").mode("overwrite").saveAsTable("silver_macro_features")
print(f"Created silver_macro_features: {sdf_silver_macro.count()} quarterly macro observations.")
display(sdf_silver_macro.tail(5))

StatementMeta(, 1e5b9eeb-8936-4921-af84-2d7f59763fe5, 9, Finished, Available, Finished, False)

Created silver_macro_features: 32 quarterly macro observations.


SynapseWidget(Synapse.DataFrame, f0e85f82-8a92-4d81-9acd-627c06b7008d)